In [1]:
import sys
sys.path.append("../")

from pathlib import Path

from datasets import Dataset
from peft import PeftModel

from src.evaluating import Evaluator
from src.llm.loading import Loader as LLMLoader
from src.ner.masking import apply_masking, labels_to_entities
from src.ner.merge_spans import merge_spans
from src.reporting import write_baseline_report
from src.ner.loading import Loader as NERLoader
from src.ner.models import Model
from src.ner.processing import id2label, label2id, label_list


/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = Model(
    name="microsoft/deberta-v3-base",
    label_list=label_list,
    id2label=id2label,
    label2id=label2id,
)

model.model = model.peft_model("../ner_model")
model.name = f"{model.name} (fine-tuned NER)"
model.name


The tokenizer you are loading from '/Users/ayeustsihneyeu/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 41104.35it/s]
DebertaV2ForTokenClassification LOAD REPORT from: /Users/ayeustsihneyeu/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_he

'microsoft/deberta-v3-base (fine-tuned NER)'

In [3]:
llm_loader = LLMLoader(path=Path("../data/data.json"), test_k=0.2, seed=42)
_, tests = llm_loader.train_test_split()

ner_loader = NERLoader(path=Path("../data/entities.json"), test_k=0.2, seed=42)
ner_dataset = ner_loader.load()
entities_by_text = {row["text"]: row["entities"] for row in ner_dataset}

matched_rows = [
    {
        "input": row["input"],
        "target": row["target"],
        "entities": entities_by_text[row["input"]],
    }
    for row in tests
    if row["input"] in entities_by_text
]

tests = Dataset.from_list(matched_rows)
len(tests)


10

In [4]:
def predict_masked_text(text: str) -> tuple[str, list[dict]]:
    prediction = model.predict(text)
    predicted_entities = labels_to_entities(prediction["labels"], prediction["offsets"])
    predicted_entities = merge_spans(predicted_entities)
    masked_text = apply_masking(text, predicted_entities)
    return masked_text, predicted_entities


In [5]:
result = []
for test in tests:
    masked_prediction, predicted_entities = predict_masked_text(test["input"])
    result.append({
        "input": test["input"],
        "prediction": masked_prediction,
        "target": test["target"],
        "predicted_entities": predicted_entities,
    })

len(result)


`use_return_dict` is deprecated! Use `return_dict` instead!


10

In [6]:
evaluator = Evaluator(
    predictions=[row["prediction"] for row in result],
    references=[row["target"] for row in result],
)

metrics = evaluator.evaluate()
metrics


{'samples': 10,
 'exact_match': 0.0,
 'masking_recall': 0.9705882352941176,
 'over_masking_rate': 0.029411764705882353,
 'text_preservation': 0.9331160029442707}

In [7]:
report_path = write_baseline_report(
    model_name=model.name,
    metrics=metrics,
    predictions=result,
    output_path="../reports/09_ner_eval.md",
    preview_size=len(result),
)

print(f"NER report saved to {report_path}")

preview = result[:3]
for index, row in enumerate(preview, start=1):
    print()
    print(f"Sample {index}")
    print("INPUT:", row["input"])
    print("PREDICTION:", row["prediction"])
    print("TARGET:", row["target"])


NER report saved to ../reports/09_ner_eval.md

Sample 1
INPUT: Hello Mrs. Walker, can you check why my recent transfer of $250 hasn't been processed? My account number is 567890 and my email is megan.walker@example.com.
PREDICTION: Hello Mrs[PREFIX] Walker, can you check why my recent transfer of $[AMOUNT] hasn't been processed? My account number is 567[ACCOUNT_NUMBER] and my email is me[EMAIL].
TARGET: Hello [PREFIX], can you check why my recent transfer of [AMOUNT] hasn't been processed? My account number is [ACCOUNT_NUMBER] and my email is [EMAIL].

Sample 2
INPUT: Dear Ms. Garcia, I need to report a lost debit card. My phone number is (555) 789-0123 and my account number is 678901.
PREDICTION: Dear Ms[PREFIX] Garcia, I need to report a lost debit card. My phone number is ([PHONE_NUMBER] and my account number is 678[ACCOUNT_NUMBER].
TARGET: Dear [PREFIX], I need to report a lost debit card. My phone number is [PHONE_NUMBER] and my account number is [ACCOUNT_NUMBER].

Sample 3
INPUT: